# 05. Quick Damage Seg Inference — external CarDD YOLOv11

Быстрый notebook для прогона внешнего checkpoint `car-dd-segmentation-yolov11-best.pt` на одной картинке и сохранения overlay + JSON.

In [ ]:
!pip install -q ultralytics pillow numpy

In [ ]:
import json
from pathlib import Path

import numpy as np
from PIL import Image
from ultralytics import YOLO

In [ ]:
PROJECT_ROOT = Path.cwd()
for cand in [PROJECT_ROOT] + list(PROJECT_ROOT.parents):
    if (cand / 'ml').exists() and (cand / 'notebooks').exists():
        PROJECT_ROOT = cand
        break

WEIGHTS_PATH = PROJECT_ROOT / 'ml' / 'damage_seg' / 'weights' / 'external' / 'car-dd-segmentation-yolov11-best.pt'
METADATA_PATH = PROJECT_ROOT / 'ml' / 'damage_seg' / 'weights' / 'external' / 'metadata.json'
OUTPUT_DIR = PROJECT_ROOT / 'ml' / 'damage_seg' / 'reports' / 'quick_inference'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_PATH = PROJECT_ROOT / 'sample.jpg'  # поменяй на свою картинку
CONF = 0.25
IMGSZ = 1024

print('WEIGHTS_PATH =', WEIGHTS_PATH)
print('METADATA_PATH =', METADATA_PATH)
print('OUTPUT_DIR =', OUTPUT_DIR)
print('IMAGE_PATH =', IMAGE_PATH)

In [ ]:
meta = json.loads(METADATA_PATH.read_text(encoding='utf-8')) if METADATA_PATH.exists() else {}
class_aliases = meta.get('class_aliases', {})

def normalize_classes(raw_classes):
    return [class_aliases.get(cls, cls.replace(' ', '_')) for cls in raw_classes]

def detect_device():
    try:
        import torch
        if torch.cuda.is_available():
            return '0'
        if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
            return 'mps'
    except Exception:
        pass
    return 'cpu'

def serialize_result(result, damage_classes, image_size):
    img_w, img_h = image_size
    rows = []
    if result.boxes is None or len(result.boxes) == 0:
        return rows
    for i in range(len(result.boxes)):
        cls_idx = int(result.boxes.cls[i].item())
        if cls_idx >= len(damage_classes):
            continue
        xyxy = result.boxes.xyxy[i].cpu().numpy()
        x1 = float(xyxy[0]) / img_w
        y1 = float(xyxy[1]) / img_h
        x2 = float(xyxy[2]) / img_w
        y2 = float(xyxy[3]) / img_h
        polygon_json = None
        if result.masks is not None and i < len(result.masks):
            mask = result.masks[i]
            if mask.xy is not None and len(mask.xy) > 0:
                poly_px = mask.xy[0] if isinstance(mask.xy, list) else mask.xy
                if hasattr(poly_px, 'cpu'):
                    poly_px = poly_px.cpu().numpy()
                elif not isinstance(poly_px, np.ndarray):
                    poly_px = np.array(poly_px)
                polygon_json = [[round(float(pt[0]) / img_w, 6), round(float(pt[1]) / img_h, 6)] for pt in poly_px]
        if polygon_json is None:
            polygon_json = [[round(x1, 6), round(y1, 6)], [round(x2, 6), round(y1, 6)], [round(x2, 6), round(y2, 6)], [round(x1, 6), round(y2, 6)]]
        rows.append({
            'damage_type': damage_classes[cls_idx],
            'confidence': round(float(result.boxes.conf[i].item()), 4),
            'bbox_norm': {'x1': round(x1, 6), 'y1': round(y1, 6), 'x2': round(x2, 6), 'y2': round(y2, 6)},
            'polygon_json': polygon_json,
        })
    return rows

In [ ]:
assert WEIGHTS_PATH.exists(), f'Missing weights: {WEIGHTS_PATH}'
assert IMAGE_PATH.exists(), f'Missing image: {IMAGE_PATH}'

model = YOLO(str(WEIGHTS_PATH))
raw_names = getattr(model.model, 'names', None)
if isinstance(raw_names, dict):
    damage_classes = [raw_names[i] for i in sorted(raw_names)]
elif isinstance(raw_names, list):
    damage_classes = raw_names
else:
    damage_classes = meta.get('damage_classes', [])
damage_classes = normalize_classes(damage_classes)

device = detect_device()
image = Image.open(IMAGE_PATH).convert('RGB')
results = model.predict(source=image, imgsz=IMGSZ, conf=CONF, device=device, verbose=False)
result = results[0]
detections = serialize_result(result, damage_classes, image.size)
overlay = Image.fromarray(result.plot())

stem = IMAGE_PATH.stem
json_path = OUTPUT_DIR / f'{stem}_predictions.json'
overlay_path = OUTPUT_DIR / f'{stem}_overlay.jpg'
json_path.write_text(json.dumps({'image': str(IMAGE_PATH), 'weights': str(WEIGHTS_PATH), 'device': device, 'imgsz': IMGSZ, 'conf': CONF, 'damage_classes': damage_classes, 'detections': detections}, indent=2, ensure_ascii=False), encoding='utf-8')
overlay.save(overlay_path, format='JPEG', quality=95)

print('detections =', len(detections))
print('json_path =', json_path)
print('overlay_path =', overlay_path)
detections[:5]

In [ ]:
from IPython.display import display
display(Image.open(overlay_path))